# Part 3: Debugging training

A network that is broken usually does not crash. It trains, prints numbers, and gives poor results. This notebook is a set of tests that tell you which kind of broken you have. Run them in this order: shapes, one batch, the learning rate, then overfitting.

In [ ]:
import warnings

import torch
from torch import nn

from churn_data import make_splits
from churn_net import ChurnNet

splits = make_splits(42)
x, y = splits.train.x, splits.train.y

## 1. Shapes and types

`ChurnNet` returns one score per customer with shape `(N,)`. A model that ends in `Linear(n, 1)` and forgets `.squeeze(1)` returns `(N, 1)`. Against targets of shape `(N,)`, PyTorch broadcasts the pair into an `N` by `N` table instead of stopping. The loss gets computed over the wrong comparison, and training still runs.

In [ ]:
torch.manual_seed(0)
net = ChurnNet(9)
pred, target = net(x[:64]), y[:64]
print("pred", tuple(pred.shape), "target", tuple(target.shape))

mse = nn.MSELoss()
print("matching shapes:", round(mse(pred, target).item(), 4))
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print("(N, 1) against (N,):", round(mse(pred.unsqueeze(1), target).item(), 4))
    print("warning:", str(caught[0].message)[:90], "...")

Some mismatches stop with an error. Read the message, because it names the problem:

In [ ]:
for label, call in {
    "integer labels": lambda: nn.BCEWithLogitsLoss()(pred, target.long()),
    "extra dimension": lambda: nn.BCEWithLogitsLoss()(pred.unsqueeze(1), target),
}.items():
    try:
        call()
    except Exception as error:
        print(f"{label}: {type(error).__name__}: {str(error)[:80]}")

## 2. Can the model memorize one batch?

Take 32 rows and train on only those. A healthy model and loop must be able to drive the loss close to zero, because memorizing 32 rows is easy. If it cannot, the problem is in the code and not in the data. Each variant below breaks one thing.

In [ ]:
def overfit_one_batch(model, xb, yb, steps=300, lr=1e-2, step=True, zero=True):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    first = None
    for _ in range(steps):
        if zero:
            optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        if step:
            optimizer.step()
        if first is None:
            first = loss.item()
    return round(first, 4), round(loss.item(), 4)


xb, yb = x[:32], y[:32]
cases = {
    "healthy": {},
    "optimizer.step() missing": {"step": False},
    "zero_grad() missing": {"zero": False},
    "learning rate 1e-6": {"lr": 1e-6},
}
for label, options in cases.items():
    torch.manual_seed(0)
    first, last = overfit_one_batch(ChurnNet(9, hidden=64), xb, yb, **options)
    print(f"{label:26s} first {first}  after 300 steps {last}")

The healthy model goes from 0.66 to nearly 0. With no `step`, nothing moves. With no `zero_grad`, gradients pile up from step to step and the loss ends far from zero. A learning rate that is too small barely moves in 300 steps.

## 3. The learning rate on real training

Now train for 30 epochs on the full training set and compare final validation losses.

In [ ]:
from churn_data import make_loaders


def train_for(lr, epochs=30, seed=0):
    torch.manual_seed(seed)
    train_loader, _, _ = make_loaders(splits)
    net = ChurnNet(9)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(epochs):
        net.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss_fn(net(xb), yb).backward()
            optimizer.step()
    net.eval()
    with torch.no_grad():
        return loss_fn(net(splits.val.x), splits.val.y).item()


for lr in (1e-5, 1e-3, 3e-3, 1e-2, 1.0, 10.0):
    print(f"Adam lr {lr:<7} validation loss {train_for(lr):.3f}")

A rate of 1e-5 barely moves from the starting loss of about 0.70, so the curve looks flat. Rates from 1e-3 to 1e-2 all reach about 0.33 to 0.35. A rate of 1.0 or 10 does not blow up on this small, scaled problem, but it ends well above the others. On a bigger network the same mistake often ends in `nan`. Start near 1e-3 for Adam and change one thing at a time.

## 4. Overfitting

Give a large network only 150 training rows, and train it on the full batch. Watch the validation loss.

In [ ]:
tiny_x, tiny_y = x[:150], y[:150]


def curve(model, epochs=300, weight_decay=0.0):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=weight_decay)
    train_losses, val_losses = [], []
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(tiny_x), tiny_y)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_losses.append(loss_fn(model(splits.val.x), splits.val.y).item())
        train_losses.append(loss.item())
    return train_losses, val_losses


for label, hidden, decay in (("big net", 128, 0.0), ("big net + weight decay", 128, 1e-2), ("small net", 8, 0.0)):
    torch.manual_seed(0)
    train_losses, val_losses = curve(ChurnNet(9, hidden=hidden), weight_decay=decay)
    best = min(range(300), key=val_losses.__getitem__)
    print(f"{label:24s} best val {val_losses[best]:.3f} at epoch {best + 1:3d}   final train {train_losses[-1]:.3f}  final val {val_losses[-1]:.3f}")

The big network's training loss falls to almost zero while its validation loss climbs from 0.39 to over 3. It memorized 150 rows. The three remedies you have are **stop at the best epoch**, **a smaller network**, and **weight decay**, which penalizes large weights. Here weight decay keeps the final validation loss near 0.5 instead of 3.1, and the small network does best. More data would beat all three.

## Your turn

Add `nn.Dropout(0.3)` after each hidden layer of the big network and rerun the last cell. Remember `model.eval()` when you measure.